# 07 reachability

Builds the coverage matrices the optimiser needs, one row per water point and
one column per demand cell.

A single cost accumulation seeded with every point returns, for each cell, the
time to the nearest point but not which point that is. That is enough to
describe present accessibility and not enough to decide which subset to repair,
so one accumulation is run per point, restricted to a window slightly larger
than the threshold allows on foot.

Writes `reach_cand.npz` and `reach_supp.npz`, both sparse.

In [11]:
import sys
from pathlib import Path

# config.py sits beside the notebooks, so the working directory is enough. If a
# notebook is run from elsewhere, walk up until it is found.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
from config import *


## 1. Inputs

In [12]:
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.transform import rowcol, xy
from rasterio.warp import reproject, Resampling
from scipy import sparse
from skimage.graph import MCP_Geometric
from rasterio.features import rasterize

# Threshold in minutes, one way. Set to THRESHOLD_SENSITIVITY for the 30 minute
# run and change the two output paths in the last cell.
THRESHOLD = THRESHOLD_SENSITIVITY

demand = pd.read_csv(DEMAND_CELLS)
cand = pd.read_csv(KIKUUBE_CAND)
supp = pd.read_csv(KIKUUBE_SUPPLY)
print(f"demand cells {len(demand):,}, candidates {len(cand)}, supply {len(supp)}")
print(f"threshold {THRESHOLD:.0f} minutes one way")

demand cells 100,619, candidates 320, supply 1209
threshold 30 minutes one way


## 2. Rebuild the grid

06 clipped the population raster to the district and recorded the row and column
of every cell it kept. Repeating the same clip here recovers the identical grid,
which is asserted rather than assumed.

In [13]:
sub = gpd.read_file(SUBCOUNTY_GJ).to_crs(CRS_GEO)

# Cost surface gets a 3 km buffer so least-cost paths may leave and re-enter
# the district; demand cells remain strictly inside. 3000 m exceeds the 2.5 km
# a 30-minute walk can possibly cover, so both thresholds are safe.
BUFFER_M = 3000
geom_buffered = (sub.dissolve().to_crs(CRS_METRIC)
                    .geometry.buffer(BUFFER_M)
                    .to_crs(CRS_GEO))

with rasterio.open(WORLDPOP_TIF) as src:
    arr, T = mask(src, geom_buffered, crop=True, nodata=np.nan, filled=True)
H, W = arr[0].shape

# Re-locate every demand cell on the buffered grid from its coordinates.
new_r, new_c = rowcol(T, demand.lon.values, demand.lat.values)
new_r, new_c = np.asarray(new_r), np.asarray(new_c)
assert (new_r >= 0).all() and (new_r < H).all(), "demand cells fall outside the buffered grid"
assert (new_c >= 0).all() and (new_c < W).all(), "demand cells fall outside the buffered grid"

chk_x, chk_y = xy(T, new_r[:200], new_c[:200])
assert np.allclose(chk_x, demand.lon.values[:200], atol=abs(T.a)), "relocation failed"
assert np.allclose(chk_y, demand.lat.values[:200], atol=abs(T.a)), "relocation failed"

print(f"buffered grid {H} x {W}, cell {abs(T.a):.6f} deg")
print(f"all {len(demand):,} demand cells located on the buffered grid")

rural = np.zeros((H, W), dtype=bool)
rural[new_r, new_c] = True
didx = -np.ones((H, W), dtype=int)
didx[new_r, new_c] = np.arange(len(demand))

buffered grid 637 x 1107, cell 0.000833 deg
all 100,619 demand cells located on the buffered grid


## 3. Cost surface

The friction surface is resampled onto the population grid so that both share
one indexing scheme. This adds no information, it removes the need to translate
between two grids.

In [14]:
with rasterio.open(FRICTION_TIF) as src:
    fr = np.empty((H, W), "float64")
    reproject(rasterio.band(src, 1), fr,
              dst_transform=T, dst_crs=CRS_GEO,
              resampling=Resampling.nearest)

# Friction is minutes to cross one metre, so the cost of a cell is that value
# times the cell width. The width is corrected for latitude.
CELL = abs(T.a) * 111320 * np.cos(np.radians(demand.lat.mean()))
cost = np.where(np.isfinite(fr) & (fr > 0), fr * CELL, np.inf)

print(f"cell width {CELL:.0f} m at this latitude")
print(f"cost per cell, median {np.median(cost[np.isfinite(cost)]):.2f} minutes")
print(f"cells with no usable friction value {int((~np.isfinite(cost)).sum()):,}")

cell width 93 m at this latitude
cost per cell, median 1.11 minutes
cells with no usable friction value 0


In [15]:
# Verify the buffer zone carries real friction values, not inf.
inside = rasterize([(g, 1) for g in sub.dissolve().geometry],
                   out_shape=(H, W), transform=T, fill=0, dtype="uint8").astype(bool)
buffer_zone = np.isfinite(cost) & ~inside
print(f"buffer cells with usable friction {int(buffer_zone.sum()):,}")
assert buffer_zone.sum() > 0, "buffer zone has no usable friction - check FRICTION_TIF extent"

buffer cells with usable friction 355,264


## 4. Window size

Each accumulation runs inside a window around its seed. The radius is set from
the fastest walking speed on the surface, which is the largest distance the
threshold could possibly buy, so nothing reachable falls outside it.

In [16]:
FASTEST = 0.012  # min/m: lower bound used to make the local window safe

actual_fastest = np.nanmin(fr[np.isfinite(fr) & (fr > 0)])

assert FASTEST <= actual_fastest + 1e-12, (
    f"FASTEST={FASTEST}, but the computational friction grid contains "
    f"a lower positive value ({actual_fastest}). The window may be too small."
)

PAD = int(np.ceil(THRESHOLD / FASTEST / CELL)) + 2

print(f"verified minimum friction in computational grid: {actual_fastest:.6f} min/m")
print(f"window radius {PAD} cells, about {PAD * CELL / 1000:.1f} km")
print("minimum-friction lower bound verified; no threshold-reachable cell is cut off")

verified minimum friction in computational grid: 0.012000 min/m
window radius 29 cells, about 2.7 km
minimum-friction lower bound verified; no threshold-reachable cell is cut off


## 5. One accumulation per point

In [17]:
def reach_pairs(df, label):
    r0, c0 = rowcol(T, df["#lon_deg"].values, df["#lat_deg"].values)
    r0, c0 = np.asarray(r0), np.asarray(c0)
    R, C = [], []
    skipped = 0
    for k in range(len(df)):
        if not (0 <= r0[k] < H and 0 <= c0[k] < W):
            skipped += 1
            continue
        a, b = max(0, r0[k] - PAD), min(H, r0[k] + PAD + 1)
        c, d = max(0, c0[k] - PAD), min(W, c0[k] + PAD + 1)
        m = MCP_Geometric(cost[a:b, c:d], sampling=(1, 1))
        tt, _ = m.find_costs([[r0[k] - a, c0[k] - c]])
        sel = (tt <= THRESHOLD) & rural[a:b, c:d]
        ii = np.where(sel)
        R.extend([k] * len(ii[0]))
        C.extend(didx[ii[0] + a, ii[1] + c])
        if (k + 1) % 200 == 0:
            print(f"  {label}: {k + 1} of {len(df)}")
    if skipped:
        print(f"  {label}: {skipped} points fall outside the grid")
    return np.asarray(R), np.asarray(C)


cr, ccl = reach_pairs(cand, "candidates")
sr, scl = reach_pairs(supp, "supply")
print(f"\nreach pairs, candidates {len(cr):,}, supply {len(sr):,}")

  candidates: 200 of 320
  supply: 200 of 1209
  supply: 400 of 1209
  supply: 600 of 1209
  supply: 800 of 1209
  supply: 1000 of 1209
  supply: 1200 of 1209

reach pairs, candidates 271,156, supply 1,224,030


## 6. Store and inspect

A candidate reaching no demand cell can never enter a solution, either because
the cells around it were classified as urban or because they carry no
population. The effective candidate set is smaller than the nominal one.

In [18]:
# Output paths follow the threshold, so the two runs cannot overwrite each other.
suffix = "" if THRESHOLD == THRESHOLD_MIN else f"_{int(THRESHOLD)}"
path_c = OUT / f"reach_cand{suffix}.npz"
path_s = OUT / f"reach_supp{suffix}.npz"

A_cand = sparse.coo_matrix(
    (np.ones(len(cr), dtype="int8"), (cr, ccl)),
    shape=(len(cand), len(demand))).tocsr()
A_supp = sparse.coo_matrix(
    (np.ones(len(sr), dtype="int8"), (sr, scl)),
    shape=(len(supp), len(demand))).tocsr()

sparse.save_npz(path_c, A_cand)
sparse.save_npz(path_s, A_supp)

reached_c = np.asarray(A_cand.sum(axis=0)).ravel() > 0
reached_s = np.asarray(A_supp.sum(axis=0)).ravel() > 0
dead = int((np.asarray(A_cand.sum(axis=1)).ravel() == 0).sum())

print(f"threshold {THRESHOLD:.0f} minutes, written to {path_c.name} and {path_s.name}\n")
print(f"candidate matrix {A_cand.shape}, density {A_cand.nnz / np.prod(A_cand.shape):.5f}")
print(f"supply matrix    {A_supp.shape}, density {A_supp.nnz / np.prod(A_supp.shape):.5f}\n")
print(f"cells within reach of a functioning point  {reached_s.sum():,} "
      f"({demand['pop'][reached_s].sum():,.0f} people)")
print(f"cells within reach of a candidate          {reached_c.sum():,} "
      f"({demand['pop'][reached_c].sum():,.0f} people)")
print(f"cells reached by neither                   {(~reached_c & ~reached_s).sum():,} "
      f"({demand['pop'][~reached_c & ~reached_s].sum():,.0f} people)")
print(f"\ncandidates reaching no demand cell {dead} of {len(cand)}")

threshold 30 minutes, written to reach_cand_30.npz and reach_supp_30.npz

candidate matrix (320, 100619), density 0.00842
supply matrix    (1209, 100619), density 0.01006

cells within reach of a functioning point  93,388 (228,408 people)
cells within reach of a candidate          73,719 (186,010 people)
cells reached by neither                   6,339 (9,659 people)

candidates reaching no demand cell 0 of 320


To produce the sensitivity run, set `THRESHOLD = THRESHOLD_SENSITIVITY` in the
first cell, change the two output paths, and rerun. Nothing else changes.